# nb_02_silver_customers

**Layer**: Silver | **Source**: `lh_bronze_insurance.customers_raw` | **Target**: `lh_silver_insurance.customers`

**Purpose**: Cleanse and conform customer data from Bronze to Silver.

**Data Quality Rules**:
- Cast `date_of_birth` and `customer_since` to DateType
- Require non-null: `customer_id`, `first_name`, `last_name`
- Deduplicate on `customer_id` (keep latest `_ingested_at`)
- Trim whitespace from string columns
- Rejects → `customers_quarantine` with `_rejection_reason`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit, row_number, when, coalesce
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, DateType

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_02_silver_customers").getOrCreate()

print("nb_02_silver_customers - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("customers_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & trim strings
# ============================================================
df_typed = df_bronze \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("email", trim(col("email"))) \
    .withColumn("phone", trim(col("phone"))) \
    .withColumn("date_of_birth", to_date(col("date_of_birth"), "yyyy-MM-dd")) \
    .withColumn("address", trim(col("address"))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("state", trim(col("state"))) \
    .withColumn("zip_code", trim(col("zip_code"))) \
    .withColumn("customer_since", to_date(col("customer_since"), "yyyy-MM-dd"))

print("Type casting complete.")
df_typed.printSchema()

In [ ]:
# ============================================================
# Step 3: Validate required fields & route rejects
# ============================================================
REQUIRED_FIELDS = ["customer_id", "first_name", "last_name"]

# Build rejection reason
rejection_conditions = []
for field in REQUIRED_FIELDS:
    rejection_conditions.append(
        when(
            (col(field).isNull()) | (col(field) == ""),
            lit(f"null_{field}")
        )
    )

df_validated = df_typed.withColumn(
    "_rejection_reason",
    coalesce(*rejection_conditions)
)

# Split into valid and rejected
df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on business key (customer_id)
# ============================================================
window_spec = Window.partitionBy("customer_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} rows ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write to Silver & Quarantine
# ============================================================
# Select final Silver columns (drop Bronze metadata)
silver_columns = [
    "customer_id", "first_name", "last_name", "email", "phone",
    "date_of_birth", "address", "city", "state", "zip_code", "customer_since"
]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

# Write Silver table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customers")
print(f"✓ Written {deduped_count:,} rows to customers")

# Write quarantine table
if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("customers_quarantine")
    print(f"✓ Written {rejected_count:,} rows to customers_quarantine")
else:
    print("No rejected rows — quarantine table not written.")

In [ ]:
# ============================================================
# Validation Summary
# ============================================================
print("\n" + "=" * 60)
print("SILVER CUSTOMERS - PROCESSING SUMMARY")
print("=" * 60)
print(f"  Bronze input:        {bronze_count:>8,}")
print(f"  Rejected (nulls):    {rejected_count:>8,}")
print(f"  Duplicates removed:  {dupes_removed:>8,}")
print(f"  Silver output:       {deduped_count:>8,}")
print(f"  Data quality rate:   {(deduped_count/bronze_count*100):.1f}%")
print()

# Verify by reading back
silver_readback = spark.table("customers").count()
assert silver_readback == deduped_count, f"Row count mismatch: {silver_readback} vs {deduped_count}"
print(f"✓ Verification passed: {silver_readback:,} rows in Silver")

# Show sample
print("\n--- Sample Silver Customers ---")
spark.table("customers").show(5, truncate=25)